# AuK language + duration tests (machine measurements only)
Runs on branch limits-language-tests. Run cells in order.

In [ ]:
# @title Step 1. Install tools and the backend v0.6.6 (voice enhancer + transcript editor; no inference)
# Downloads the pinned backend (v0.6.6) from the GitHub repo at a fixed commit, so later repo
# changes can't change what this notebook installs. Repo: github.com/advayc/auk-production-backend.
# Qwen2.5-Omni-3B conditioner remains non-commercial.
import sys, subprocess, pathlib, os
if not (3,11) <= sys.version_info[:2] <= (3,13):
    raise RuntimeError("Use a Python 3.11-3.13 Colab runtime.")
WORK_DIR = "/content/auk-production"  # @param {type:"string"}
COMMERCIAL_USE = False  # @param {type:"boolean"}
COMMERCIAL_LICENSE_CONFIRMED = False  # @param {type:"boolean"}
if COMMERCIAL_USE and not COMMERCIAL_LICENSE_CONFIRMED: raise RuntimeError("Obtain commercial model-license clearance first.")
ROOT=pathlib.Path(WORK_DIR).resolve()
if str(ROOT) in ("/", "/content", "/tmp", str(pathlib.Path.home())): raise ValueError("Use a dedicated working directory")
ROOT.mkdir(parents=True,exist_ok=True);os.umask(0o077)
subprocess.run(["apt-get","-qq","update"],check=True)
subprocess.run(["apt-get","-qq","install","-y","ca-certificates","curl","ffmpeg","libgomp1","libssl-dev","git","cmake","build-essential","python3-venv"],check=True)
BACKEND_REPO="https://github.com/advayc/auk-production-backend.git"
BACKEND_BRANCH="limits-language-tests"  # test branch; not the pinned release
REPO_DIR=ROOT/"auk-production-backend"
if REPO_DIR.is_dir(): shutil_rm=__import__("shutil").rmtree(REPO_DIR)
subprocess.run(["git","clone","--quiet","--branch",BACKEND_BRANCH,BACKEND_REPO,str(REPO_DIR)],check=True)
BACKEND_COMMIT=subprocess.run(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],check=True,capture_output=True,text=True).stdout.strip()
BACKEND_DIR=REPO_DIR/"backend"
import shutil; shutil.rmtree(BACKEND_DIR/"build", ignore_errors=True)  # never reuse stale build files
subprocess.run([sys.executable,"-m","pip","install","--no-deps",str(BACKEND_DIR)],check=True)
if "AUK_WORKER" in globals():AUK_WORKER.close();del AUK_WORKER
for name in list(sys.modules):
    if name=="auk_reliability" or name.startswith("auk_reliability."):del sys.modules[name]
import auk_reliability
RUNS_DIR = "/content/auk-runs"  # local output folder for this session
sys.path.insert(0,str(BACKEND_DIR))
print("Installed",auk_reliability.__version__,"from branch",BACKEND_BRANCH,"@",BACKEND_COMMIT[:12])


In [ ]:
# @title Step 2. Set up the runtime and download the models (downloads, no generation)
from auk_reliability.core import Limits, gpu_details, setup_runtime, setup_models, command, atomic_json
import site, os, subprocess, sys
LIMITS=Limits();GPU=gpu_details();print(GPU)
if GPU["compute_capability"] not in ("7.5","8.9"):raise RuntimeError("The pinned prebuilt serves T4 (7.5) and L4 (8.9) only. Other GPUs require the explicit source-build path.")
if GPU["compute_capability"]=="8.9":print("L4 (SM89) prebuilt: on-VM qualification = ldd + --help + this session's QA-gated edits.")
if GPU["total_mib"]<14000:raise RuntimeError("At least roughly 16 GiB total VRAM required")
# audio.cpp SM75 release uses CUDA 12.8, including on Colab images that ship CUDA 13.
subprocess.run([sys.executable,"-m","pip","install","--quiet","nvidia-cublas-cu12==12.8.4.1","nvidia-cuda-runtime-cu12==12.8.90","nvidia-cufft-cu12==11.3.3.83"],check=True)
search=set(site.getsitepackages()+[site.getusersitepackages()]);libdirs=[]
for root in search:
    for component in ("cublas","cuda_runtime","cufft"):
        p=pathlib.Path(root)/"nvidia"/component/"lib"
        if p.is_dir():libdirs.append(str(p))
os.environ["LD_LIBRARY_PATH"]=":".join(libdirs+[os.environ.get("LD_LIBRARY_PATH","")])
MEMORY_SAVER=True
BINARY,RUNTIME_LOCK=setup_runtime(ROOT,mode="t4_prebuilt",gpu=GPU)
link_check=command(["ldd",str(BINARY)],timeout=30)
if "not found" in link_check:raise RuntimeError("Runtime has unresolved libraries: "+link_check)
MODEL_DIR,MODEL_LOCK=setup_models(ROOT,["base_f32","base_q8","flash_f16"])
PROVENANCE={"runtime":RUNTIME_LOCK,"model":MODEL_LOCK,"environment":{"python":sys.version,"gpu":GPU,"nvidia_smi":command(["nvidia-smi"]),"ffmpeg":command(["ffmpeg","-version"]).splitlines()[0],"backend_commit":BACKEND_COMMIT}}
atomic_json(ROOT/"environment.json",PROVENANCE)
print("Ready. No model generation started. First request includes cold startup.")


In [ ]:
# @title Step 3. Start the worker and the QA checks (downloads, no generation)
import pathlib
ROOT = pathlib.Path(ROOT) if isinstance(ROOT, str) else ROOT
ASR_LANGUAGE = "auto"  # @param {type:"string"}
SETUP_SHARED_WORKER = True  # @param {type:"boolean"}
ENABLE_SPEAKER_QA = True  # @param {type:"boolean"}
GPU_HOURLY_RATE_USD = -1.0  # @param {type:"number"}
if SETUP_SHARED_WORKER:
    _need = [n for n in ("ROOT","LIMITS","GPU","MEMORY_SAVER","ASR_LANGUAGE","BINARY","MODEL_DIR") if n not in globals()]
    if _need:
        raise RuntimeError("Run steps 1 and 2 first (missing " + ", ".join(_need) + "). ")
    if "PROVENANCE" not in globals():
        PROVENANCE = {"runtime": RUNTIME_LOCK, "model": MODEL_LOCK, "environment": {"python":sys.version}}
    if "auk_reliability" not in sys.modules:
        raise RuntimeError("Run step 1 first.")
    from auk_reliability.core import LocalAuKServer, setup_asr
    from auk_reliability.colab import setup_optional_speaker_qa
    from auk_reliability.qa import SemanticQAGate
    from auk_reliability.worker import ReliabilityWorker, configure_default_worker, edit_audio
    from auk_reliability.acceptance import configure_acceptance_suite, run_acceptance_suite, benchmark_profiles, recommend_profiles
    if "AUK_WORKER" in globals(): AUK_WORKER.close()
    shared_asr, shared_asr_fingerprint = setup_asr(pathlib.Path(ROOT) / "shared-qa", LIMITS, ASR_LANGUAGE)
    speaker_callback, speaker_fingerprint, speaker_setup = setup_optional_speaker_qa(
        pathlib.Path(ROOT) / "shared-speaker-qa", enabled=ENABLE_SPEAKER_QA)
    print("Speaker QA:", speaker_setup)
    if speaker_callback is None:
        raise RuntimeError("Speaker QA unavailable. Fix setup before reliability runs; unavailable identity QA never passes.")
    transport = LocalAuKServer(BINARY, MODEL_DIR, ROOT / "shared-runtime", LIMITS, MEMORY_SAVER)
    runtime_profile = {**PROVENANCE, "gpu": GPU, "simulation": False,
        "asr_fingerprint": shared_asr_fingerprint, "speaker_fingerprint": speaker_fingerprint, "speaker_setup": speaker_setup}
    if GPU_HOURLY_RATE_USD >= 0: runtime_profile["gpu_hourly_rate_usd"] = GPU_HOURLY_RATE_USD
    AUK_WORKER = ReliabilityWorker(transport, ROOT / "shared-requests",
        semantic_gate=SemanticQAGate(shared_asr, speaker_callback), runtime_profile=runtime_profile)
    configure_default_worker(AUK_WORKER)
    print("Shared worker configured. No generation started. Attribute QA still needs human review.")
else:
    print("Shared worker setup is off. Enable when ready for QA downloads/install.")


In [ ]:
# @title Language tests (fetch FLEURS clips, run enhance / speed / delete per language)
import pathlib, json
from limit_tests.fetch_data import fleurs, librispeech_long
from limit_tests.run_tests import run_language_tests
DATA=pathlib.Path("/content/limit-data");DATA.mkdir(exist_ok=True)
OUT="/content/limit-results/lang"
clips=fleurs(DATA)
print(len(clips),"clips")
run_language_tests(AUK_WORKER, shared_asr, clips, OUT)
print("LANG DONE")

In [ ]:
# @title Duration ladder (raised limits)
import pathlib, dataclasses
from limit_tests.fetch_data import librispeech_long
from limit_tests.run_tests import run_duration_ladder
STEPS=[20,30,45,60,90,120]  # @param {type:"raw"}
OPS=("speed","delete","enhance")  # @param {type:"raw"}
BIG=dataclasses.replace(LIMITS,max_source_seconds=3700.0,max_input_seconds=1300.0,max_output_seconds=1300.0,combined_seconds=2700.0,inference_timeout=3600.0,asr_timeout=1800.0,wall_clock_budget_seconds=None)
AUK_WORKER.close()
from auk_reliability.core import LocalAuKServer
transport=LocalAuKServer(BINARY, MODEL_DIR, ROOT/"shared-runtime", BIG, MEMORY_SAVER)
AUK_WORKER=ReliabilityWorker(transport, ROOT/"shared-requests-ladder", semantic_gate=SemanticQAGate(shared_asr, speaker_callback), runtime_profile=runtime_profile, limits=BIG) if "limits" in ReliabilityWorker.__init__.__code__.co_varnames else None
print(AUK_WORKER)
longs=librispeech_long(pathlib.Path("/content/limit-data"))
print([(l["target"],l["secs"]) for l in longs])
run_duration_ladder(AUK_WORKER, longs, "/content/limit-results/dur", STEPS, OPS)
print("DUR DONE")